# Demo 6: Multi-Agent Orchestration

This demo covers the supervisor-worker architecture, LLM-driven vs. code-driven orchestration, Agent Bricks, Genie integration, and code-first vs. no-code decision frameworks.

> **Prerequisite**: Complete Demo 5 (Building Agents) first to understand agent anatomy, the reason-act-observe loop, and @function_tool before moving to multi-agent systems.

In [0]:
%sql
-- SETUP: Create catalog, schema, sample data, and UC functions
-- We create a dedicated catalog/schema for this demo with a
-- support_tickets table for multi-agent routing scenarios.
-- UC functions registered here serve as specialized agent tools.

CREATE CATALOG IF NOT EXISTS module5a_demo6;
CREATE SCHEMA IF NOT EXISTS module5a_demo6.agent_systems;

-- Sample support tickets table for multi-agent routing demos
CREATE OR REPLACE TABLE module5a_demo6.agent_systems.support_tickets (
  ticket_id    STRING NOT NULL,
  customer_id  STRING NOT NULL,
  category     STRING,
  priority     STRING,
  subject      STRING,
  description  STRING,
  status       STRING,
  created_at   TIMESTAMP
);

INSERT INTO module5a_demo6.agent_systems.support_tickets VALUES
('TKT-001', 'CUST-001', 'billing',   'high',   'Overcharged on order ORD-001', 'I was charged $249.99 but the product was on sale for $199.99', 'open', '2026-09-25 08:00:00'),
('TKT-002', 'CUST-002', 'shipping', 'medium', 'Order ORD-003 not delivered',   'My order from Sept 20 has not arrived yet',                     'open', '2026-09-25 09:30:00'),
('TKT-003', 'CUST-003', 'technical','high',   'Cannot access my account',     'I keep getting a 403 error when trying to log in',             'open', '2026-09-25 10:15:00'),
('TKT-004', 'CUST-001', 'general',   'low',    'Product recommendation',      'Can you recommend accessories for my recent purchase?',        'open', '2026-09-25 11:00:00'),
('TKT-005', 'CUST-002', 'billing',   'medium', 'Discount not applied',        'My premium discount was not applied to order ORD-008',         'open', '2026-09-25 11:45:00');

-- UC functions as specialized agent tools
CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.get_ticket(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the subject and description of a support ticket'
RETURN (SELECT max(concat_ws(' | ', subject, description))
       FROM module5a_demo6.agent_systems.support_tickets
       WHERE ticket_id = get_ticket.ticket_id);

CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.count_tickets_by_category(cat STRING)
RETURNS INT
COMMENT 'Returns the number of tickets in a given category'
RETURN SELECT count(*) FROM module5a_demo6.agent_systems.support_tickets
       WHERE category = count_tickets_by_category.cat;

CREATE OR REPLACE FUNCTION module5a_demo6.agent_systems.get_priority(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the priority of a support ticket'
RETURN (SELECT max(priority) FROM module5a_demo6.agent_systems.support_tickets
       WHERE ticket_id = get_priority.ticket_id);

SELECT * FROM module5a_demo6.agent_systems.support_tickets ORDER BY ticket_id;

ticket_id,customer_id,category,priority,subject,description,status,created_at
TKT-001,CUST-001,billing,high,Overcharged on order ORD-001,I was charged .99 but the product was on sale for .99,open,2026-09-25T08:00:00.000Z
TKT-002,CUST-002,shipping,medium,Order ORD-003 not delivered,My order from Sept 20 has not arrived yet,open,2026-09-25T09:30:00.000Z
TKT-003,CUST-003,technical,high,Cannot access my account,I keep getting a 403 error when trying to log in,open,2026-09-25T10:15:00.000Z
TKT-004,CUST-001,general,low,Product recommendation,Can you recommend accessories for my recent purchase?,open,2026-09-25T11:00:00.000Z
TKT-005,CUST-002,billing,medium,Discount not applied,My premium discount was not applied to order ORD-008,open,2026-09-25T11:45:00.000Z


## 5.2 : Supervisor-Worker (Router-Delegate) Pattern

### Concepts
The supervisor-worker pattern is the most common multi-agent architecture:

* **Supervisor**: Receives the user's question, decides which specialist to delegate to, and synthesizes the final response.
* **Workers**: Specialized agents, each with its own system prompt, tools, and domain expertise.
* **Router**: The supervisor's routing decision can be LLM-driven (the model decides) or rule-based (deterministic).

**Flow**: User -> Supervisor (routes) -> Worker (executes) -> Supervisor (synthesizes) -> User

**Common worker specializations**:
* Data Agent: Queries tables, runs SQL, returns structured data
* Research Agent: Searches knowledge base, retrieves documents
* General Agent: Handles FAQs, small talk, general questions

In [0]:
# 5.2 Demo: Supervisor routes tickets to specialized agents
# The supervisor uses ai_query to classify the ticket,
# then delegates to the appropriate specialist.

print("=== Supervisor-Worker Pattern: Ticket Routing ===")
print()

tickets = spark.sql("SELECT ticket_id, subject, category FROM module5a_demo6.agent_systems.support_tickets ORDER BY ticket_id").collect()

for t in tickets:
    ticket_id = t['ticket_id']
    subject = t['subject']
    actual_category = t['category']
    
    # Supervisor: LLM classifies the ticket
    routing = spark.sql(f"""
      SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        'Classify this support ticket into exactly one category (billing, shipping, technical, general). Ticket subject: "{subject}". Respond with just the category name.',
        modelParameters => named_struct('temperature', 0.0, 'max_tokens', 20)
      ) AS category
    """).collect()[0][0]
    
    routed = routing.strip().lower()
    
    # Worker: execute the appropriate specialist's tool
    priority = spark.sql(f"SELECT module5a_demo6.agent_systems.get_priority('{ticket_id}')").collect()[0][0]
    ticket_info = spark.sql(f"SELECT module5a_demo6.agent_systems.get_ticket('{ticket_id}')").collect()[0][0]
    
    print(f"Ticket {ticket_id}: '{subject}'")
    print(f"  Supervisor routed to: {routed} agent (actual: {actual_category})")
    print(f"  Priority: {priority}")
    print(f"  Ticket info: {ticket_info[:80]}...")
    print()

=== Supervisor-Worker Pattern: Ticket Routing ===

Ticket TKT-001: 'Overcharged on order ORD-001'
  Supervisor routed to: billing agent (actual: billing)
  Priority: high
  Ticket info: Overcharged on order ORD-001 | I was charged .99 but the product was on sale for...

Ticket TKT-002: 'Order ORD-003 not delivered'
  Supervisor routed to: shipping agent (actual: shipping)
  Priority: medium
  Ticket info: Order ORD-003 not delivered | My order from Sept 20 has not arrived yet...

Ticket TKT-003: 'Cannot access my account'
  Supervisor routed to: technical agent (actual: technical)
  Priority: high
  Ticket info: Cannot access my account | I keep getting a 403 error when trying to log in...

Ticket TKT-004: 'Product recommendation'
  Supervisor routed to: general agent (actual: general)
  Priority: low
  Ticket info: Product recommendation | Can you recommend accessories for my recent purchase?...

Ticket TKT-005: 'Discount not applied'
  Supervisor routed to: billing agent (actual: bil

## 5.5 : Agent Bricks

### Concepts
Agent Bricks is Databricks' no-code platform for building, optimizing, and governing production AI agents:

* **What it is**: A managed platform that handles the infrastructure, evaluation, and deployment of AI agents.
* **Supported types**:
  * **Knowledge Assistant**: RAG-based Q&A over your documents (Instructed Retriever)
  * **Supervisor Agent**: Multi-agent orchestrator that routes to specialized subagents
  * **Document Intelligence**: Extract, classify, and process documents at scale
  * **Custom Agents**: Build your own agent with custom tools and logic

* **Key differentiator**: No code required for setup. You specify the agent's purpose, configure it on your data, and the platform handles the rest.

> Agent Bricks is built on Unity Catalog, Model Serving, and Agent Evaluation - all the governance and observability you need for production.

## 5.6 : Agent Bricks Development Lifecycle

### Concepts
The Agent Bricks lifecycle follows a specify -> configure -> improve loop:

1. **Specify**: Define the agent's purpose, knowledge sources, and tools
   - Upload documents or point to UC volumes/tables
   - Define what the agent should and should not do

2. **Configure & Evaluate**: Test the agent on your actual data
   - The platform automatically evaluates quality using built-in judges
   - Review accuracy, groundedness, and safety scores
   - Iterate on instructions and configuration

3. **Deploy**: Publish the agent as a serving endpoint
   - Automatically versioned and governed by Unity Catalog
   - Monitor with inference tables

4. **Continuously Improve**: Use evaluation results to improve
   - Add examples to improve quality
   - Update knowledge sources
   - Re-evaluate and redeploy

> The entire lifecycle is governed: UC permissions, audit logs, and lineage tracking apply at every step.

## 5.7 : Genie Space vs. Genie Agent

### Concepts
Both provide natural-language access to data, but serve different use cases:

| Aspect | Genie Space | Genie Agent |
|---|---|---|
| **Purpose** | Curated NL interface to governed data | AI agent that can use tools and reason |
| **Scope** | Answers questions about specific tables | Can call tools, search, and take actions |
| **Setup** | Point to tables, write instructions | Configure agent with tools and prompts |
| **Use case** | "What were Q3 sales by region?" | "Analyze Q3 sales, identify anomalies, and draft a report" |
| **Autonomy** | Low - answers data questions | High - can chain tools and make decisions |

**When to use Genie Space**: When users need self-serve data exploration with NL.
**When to use Genie Agent**: When the task requires multi-step reasoning and tool use.

> Genie Spaces can be used standalone or as a specialized worker in a multi-agent system.

## 5.8 : Integrating Genie with Agents

### Concepts
Genie can serve as a specialized data worker in a multi-agent system:

* **Pattern 1: Genie as a subagent**: A supervisor agent routes data questions to a Genie Space. The Genie Space answers using its curated tables, and the supervisor includes the answer in its response.
* **Pattern 2: Genie for ad-hoc analysis**: When the user asks an unexpected data question, the supervisor delegates to Genie instead of failing.
* **Pattern 3: Standalone Genie**: No agent needed - users interact with Genie directly for data questions.

**Benefits of integration**:
* The agent handles complex reasoning and tool use
* Genie handles data queries with its optimized NL-to-SQL pipeline
* Governance is unified through Unity Catalog

> Integration pattern: Supervisor -> Genie Space (data questions) + Research Agent (knowledge) + General Agent (FAQs) -> Supervisor synthesizes response.

## 5.9 : Code-first vs. No-code Agent Platforms

### Concepts
Two approaches to building agents on Databricks:

| Approach | Tools | Best for |
|---|---|---|
| **Code-first** | OpenAI Agents SDK, LangChain, LangGraph, DSPy | Custom logic, complex tool chains, research, prototyping |
| **No-code** | Agent Bricks, Genie | Production agents, governed deployments, rapid delivery |

**Code-first advantages**:
* Full control over agent logic and flow
* Can integrate any library or API
* Ideal for research and experimentation

**No-code advantages**:
* Faster time to production
* Built-in evaluation and monitoring
* Automatic governance and compliance
* No infrastructure to manage

> Start with code-first to prototype and validate. Move to no-code (Agent Bricks) for production deployment.

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| 5.2 | Supervisor-worker ticket routing | Supervisor classifies -> delegates to specialist -> synthesizes response |
| 5.5 | Agent Bricks concepts | No-code platform: Knowledge Assistant, Supervisor Agent, Document Intelligence |
| 5.6 | Agent Bricks lifecycle | Specify -> Configure/Evaluate -> Deploy -> Continuously Improve |
| 5.7 | Genie Space vs. Genie Agent | Space: NL data queries; Agent: multi-step reasoning with tools |
| 5.8 | Integrating Genie with agents | Genie as a specialized data worker in a multi-agent system |
| 5.9 | Code-first vs. no-code | Code-first for prototyping; Agent Bricks for production |

### Key principles
* **Supervisor-worker is the default pattern**: Route, delegate, synthesize.
* **Agent Bricks for production**: No-code, governed, evaluated, monitored.

In [0]:
# CLEANUP: Drop all resources created in this demo
print("Dropping UC functions...")
for fn in ['get_ticket', 'count_tickets_by_category', 'get_priority']:
    spark.sql(f"DROP FUNCTION IF EXISTS module5a_demo6.agent_systems.{fn}")
print("  Functions dropped")

print("Dropping schema and catalog...")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo6.agent_systems CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo6 CASCADE")
print("  Schema and catalog dropped")

print("\nCleanup complete!")